# Medallion Architecture — Notebook 3: Cleaning Data into the Silver Layer

This is the third notebook in the Medallion Architecture tutorial series. Notebook 2 landed our raw sales data as-is into the Bronze table `demo.demo_sch.products_bronze`. Here we take that raw data, run **data quality checks** (duplicates, unexpected values), **clean and transform** it, and save it as the **Silver** layer table `demo.demo_sch.products_silver`.

## The Silver Layer

Silver is where raw Bronze data becomes **trustworthy**:

* **Duplicates removed** — one row per business entity (here: one row per `InvoiceNum`)
* **Unexpected values filtered** — only rows with valid countries and sensible amounts survive
* **Standardized** — consistent text casing, derived date parts
* **Enriched** — new columns useful downstream (`SalesYear`, `SalesMonth`, `ProfitAmount`)

In this notebook we:

1. Load the Bronze table `demo.demo_sch.products_bronze`
2. Run data quality checks — duplicates and unexpected values
3. Clean, transform, and write the Silver table `demo.demo_sch.products_silver`
4. Inspect and analyze the resulting Silver data

In [0]:
from pyspark.sql import functions as F

bronze_table = "demo.demo_sch.products_bronze"
bronze_df = spark.table(bronze_table)

bronze_count = bronze_df.count()
print(f"Loaded {bronze_count:,} rows from {bronze_table}")

bronze_df.printSchema()
display(bronze_df.limit(5))

## Data Quality Checks

Before writing anything to Silver, we inspect Bronze for the two most common data problems:

* **Duplicates** — the same invoice ingested more than once (e.g. a retry or a double upload)
* **Unexpected values** — countries outside the expected list, negative amounts, non-positive quantities, or tax that doesn't match the expected 8% rate

Checking *before* cleaning lets us see exactly what the cleaning steps need to fix.

In [0]:
# An invoice is our natural key: it should appear exactly once.
duplicate_invoices = (
    bronze_df
    .groupBy("InvoiceNum")
    .agg(F.count("*").alias("n_rows"))
    .filter(F.col("n_rows") > 1)
)

n_dup_invoices = duplicate_invoices.count()
dup_rows = bronze_count - bronze_df.select("InvoiceNum").distinct().count()

print(f"Invoices appearing more than once: {n_dup_invoices}")
print(f"Rows that are duplicates of another invoice: {dup_rows}")

display(duplicate_invoices.orderBy(F.desc("n_rows")).limit(20))

In [0]:
expected_countries = ["Australia", "Canada", "France", "Germany", "India", "Italy", "United States"]

# 1) Country values outside the expected list (watch for typos, casing, NULLs)
distinct_countries = bronze_df.select("Country").distinct().orderBy("Country")
display(distinct_countries)

unexpected_countries = distinct_countries.filter(~F.col("Country").isin(expected_countries))
print(f"Unexpected country values: {unexpected_countries.count()}")

# 2) Numeric sanity: negative amounts, non-positive quantities, costs above sales,
#    and tax that doesn't match the expected 8% of the sale
unexpected_values = bronze_df.select(
    F.sum((F.col("SalesAmount") < 0).cast("int")).alias("negative_sales_amounts"),
    F.sum((F.col("UnitPrice") < 0).cast("int")).alias("negative_unit_prices"),
    F.sum((F.col("OrderQuantity") <= 0).cast("int")).alias("nonpositive_quantities"),
    F.sum((F.col("TotalProductCost") < 0).cast("int")).alias("negative_costs"),
    F.sum((F.col("TotalProductCost") > F.col("SalesAmount")).cast("int")).alias("costs_exceeding_sales"),
    F.sum((F.abs(F.col("TaxAmt") / F.col("SalesAmount") - 0.08) > 0.005).cast("int")).alias("tax_not_8pct"),
)
display(unexpected_values)

# 3) NULLs in any column
null_check = bronze_df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in bronze_df.columns
])
display(null_check)

## Clean and Transform into Silver

Our checks found a clean dataset — but we still apply the **defensive cleaning rules** every Silver pipeline should have. If a bad batch arrives tomorrow, the same code handles it:

* `dropDuplicates` on `InvoiceNum` — keep one row per invoice
* Keep only rows with expected countries and valid amounts
* Standardize text (`trim` + `initcap` on `Country`)
* Enrich with `SalesYear`, `SalesMonth`, and `ProfitAmount`

In [0]:
silver_table = "demo.demo_sch.products_silver"

silver_df = (
    bronze_df
    .dropDuplicates(["InvoiceNum"])                      # dedupe on the natural key
    .filter(F.col("Country").isin(expected_countries))  # only expected countries
    .filter(
        (F.col("SalesAmount") > 0)
        & (F.col("UnitPrice") > 0)
        & (F.col("OrderQuantity") > 0)
        & (F.col("TotalProductCost") >= 0)
    )
    .withColumn("Country", F.initcap(F.trim(F.col("Country"))))  # standardize text
    .withColumn("SalesYear", F.year("OrderDate"))        # enrich: date parts
    .withColumn("SalesMonth", F.month("OrderDate"))
    .withColumn("ProfitAmount", F.round(F.col("SalesAmount") - F.col("TotalProductCost"), 2))
)

(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_table)
)

print(f"Wrote {silver_df.count():,} rows to {silver_table}")

In [0]:
# Reload from the metastore and confirm the write
silver_check = spark.table(silver_table)

print(f"Bronze rows: {bronze_count:,}")
print(f"Silver rows: {silver_check.count():,}")
print(f"Rows removed by cleaning: {bronze_count - silver_check.count():,}")
print(f"Distinct invoices in Silver: {silver_check.select('InvoiceNum').distinct().count():,}")

silver_check.printSchema()
display(silver_check.limit(10))

In [0]:
# With clean, enriched Silver data we can slice by the new columns
sales_by_year = (
    silver_check
    .groupBy("SalesYear")
    .agg(
        F.countDistinct("InvoiceNum").alias("invoices"),
        F.round(F.sum("SalesAmount"), 2).alias("total_sales"),
        F.round(F.sum("ProfitAmount"), 2).alias("total_profit"),
    )
    .orderBy("SalesYear")
)
display(sales_by_year)

profit_by_country = (
    silver_check
    .groupBy("Country")
    .agg(F.round(F.sum("ProfitAmount"), 2).alias("total_profit"))
    .orderBy(F.desc("total_profit"))
)
display(profit_by_country)

## Recap

* Loaded the raw Bronze table `demo.demo_sch.products_bronze`
* Ran data quality checks — duplicates on `InvoiceNum`, unexpected countries, negative/invalid amounts, 8% tax consistency, NULLs
* Applied defensive cleaning anyway: dedupe, validity filters, standardized countries, enrichment with `SalesYear` / `SalesMonth` / `ProfitAmount`
* Wrote and inspected the Silver table `demo.demo_sch.products_silver`

**Next up:** Notebook 4 — aggregate the Silver data into the **Gold** layer for analytics.